# Think CS in C\#

Welcome to *Think Computer Science in C#*.

This set of lecture notes (this “book”) was originally inspired by the work of Andrew N. Harrington and George Thiruvathukal [1] , serves as an introductory computer programming textbook at college level.

Designed for the students who are formally introduced to computing and computer science for the first time, this book focuses on the fundamental concepts of computer science and programming, while also incorporating key ideas from information science and information systems to provide business and management contexts to bridge students’ understanding through relevant and accessible materials.

C# has gained popularity in the industry during recent years and has become the default programming language for the .NET platform. However, this book focuses on the fundamental concepts that are common across general-purpose programming languages. Students with or without prior experience in programming languages will be able to relate their existing knowledge in computing to the materials presented, and thus strengthening their understanding of core computing and programming principles.

The book begins with basic procedural programming skills, then moves on to data structures, algorithm design, and ultimately class-based object-oriented programming, along with a selection of additional relevant topics.

Although the new AI platforms are becoming strong in automated code generation, the industry
will always have good use of strong computer scientists, system engineers, data scientists,
and cybersecurity experts, etc., and I am sure that people who hold those positions will always
need great problem-solving skills. This course and this book aim to serve that need.



<h2> C# </h2>

Albeit the overall popularity of JavaScript and Python in recent years, Java and Microsoft .NET technology remain as strong programming languages for large enterprises. In recent years especially, Microsoft has made C# the central language for their fast-evolving, open-sourced, and cross-platform .NET ecosystem. 

C# is general-purpose, high-level, multiple-paradigm, and is gaining market share as a popular programming language. As a general-purpose programming language, C# has been used in creating interactive websites, mobile apps, video games, augmented reality (AR) & virtual reality (VR), desktop applications, and back-end services. Specifically, C# is widely used in building cross-platform applications, web services, and game development. For example, the mobile game Pokemon Go is built with the Unity game engine and the Stack Overflow website is built with ASP.NET, both frameworks use C# as their programming language. 

In the annual StackOverview Developer Survey, for example, C# and Java stay as strong general-purpose languages.

In [ ]:
#r "nuget: Microsoft.Data.Analysis"
#r "nuget: Plotly.NET"
#r "nuget: Plotly.NET.Interactive"

Installed Packages Microsoft.Data.Analysis, 0.23.0 Plotly.NET, 5.1.0 Plotly.NET.Interactive, 5.0.0

In [ ]:
using System.Net.Http;
using System.Text.RegularExpressions;
using Plotly.NET;
using Plotly.NET.Interactive;
using Plotly.NET.LayoutObjects;

var surveyUrl = "https://survey.stackoverflow.co/2025";
var topLangs = new List<(string Language, double Percent)>();

var targetLanguages = new HashSet<string>(StringComparer.OrdinalIgnoreCase)
{
    "JavaScript", "HTML/CSS", "SQL", "Python", "Bash/Shell",
    "TypeScript", "Java", "C#", "C++", "PowerShell"
};

try
{
    using var http = new HttpClient();
    http.DefaultRequestHeaders.UserAgent.ParseAdd("Mozilla/5.0 (compatible; CSCSBook/1.0)");
    var html = await http.GetStringAsync(surveyUrl);

    // Parse language entries such as "JavaScript 66%" that appear in chart payload/content.
    var matches = Regex.Matches(
        html,
        @"(JavaScript|HTML/CSS|SQL|Python|Bash/Shell(?:\s*\(all shells\))?|TypeScript|Java|C#|C\+\+|PowerShell)\s*([0-9]+(?:\.[0-9]+)?)%",
        RegexOptions.IgnoreCase
    );

    foreach (Match m in matches)
    {
        var rawLanguage = m.Groups[1].Value.Trim();
        var language = rawLanguage.StartsWith("Bash/Shell", StringComparison.OrdinalIgnoreCase) ? "Bash/Shell" : rawLanguage;
        if (!targetLanguages.Contains(language)) continue;

        var pct = double.Parse(m.Groups[2].Value);
        if (!topLangs.Any(x => x.Language.Equals(language, StringComparison.OrdinalIgnoreCase)))
            topLangs.Add((language, pct));
    }
}
catch
{
    // Network or parsing failures should not break the notebook build.
}

if (topLangs.Count < 10)
{
    // Fallback top-10 values (Stack Overflow Developer Survey 2025, published summary/chart).
    topLangs = new List<(string, double)>
    {
        ("JavaScript", 66.0),
        ("HTML/CSS", 61.9),
        ("SQL", 58.6),
        ("Python", 57.9),
        ("Bash/Shell", 48.7),
        ("TypeScript", 43.8),
        ("Java", 29.5),
        ("C#", 27.9),
        ("C++", 23.6),
        ("PowerShell", 23.3),
    };
}

var ordered = topLangs
    .OrderByDescending(x => x.Percent)
    .Take(10)
    .OrderBy(x => x.Percent)
    .ToArray();

var langs = ordered.Select(x => x.Language).ToArray();
var pcts = ordered.Select(x => x.Percent).ToArray();

Chart2D.Chart.Bar<double, string, string, string, string>(
    values: pcts,
    Keys: langs,
    Name: "% Used"
 )
.WithLayout(Layout.init<string>(
    Width: 560, Height: 360,
    Title: Title.init("Top 10 Languages - Stack Overflow Survey 2025", X: 0.5, Y: 0.95),
    Margin: Margin.init<int, int, int, int, int, int>(Top: 60, Right: 30, Bottom: 50, Left: 140)
)).WithXAxis(LinearAxis.init<string, string, string, string, string, string, string, string>(
    Title: Title.init("% of Respondents")
))
.WithYAxis(LinearAxis.init<string, string, string, string, string, string, string, string>(
))
.WithConfig(Config.init(Responsive: false))
.Display();


<!-- Plotly chart will be drawn inside this DIV -->

Also, according to [TIOBE Index](https://www.tiobe.com/tiobe-index/), C# is one of the most popular languages according to the number of [search engine hits](https://www.tiobe.com/tiobe-index/programminglanguages_definition/).

In [29]:
#r "nuget: HtmlAgilityPack"
// #r "nuget: Plotly.NET"
// #r "nuget: Plotly.NET.Interactive"

Installed Packages HtmlAgilityPack, 1.12.4

In [30]:
using System.Reflection;
using System.Runtime.CompilerServices;
using HtmlAgilityPack;
using Plotly.NET;
using Plotly.NET.Interactive;
using Plotly.NET.LayoutObjects;

var web = new HtmlWeb();
var doc = web.Load("https://www.tiobe.com/tiobe-index/");

var rows = doc.DocumentNode.SelectNodes("//table[contains(@id,'top20')]//tr");

var languages = new List<string>();
var ratings = new List<double>();

foreach (var row in rows.Skip(1))
{
    var cells = row.SelectNodes("td");
    if (cells != null && cells.Count >= 6)
    {
        languages.Add(cells[4].InnerText.Trim());
        if (double.TryParse(cells[5].InnerText.Replace("%", "").Trim(), out double rating))
            ratings.Add(rating);
    }
}

var top10Languages = languages.Take(10).Reverse().ToArray();
var top10Ratings = ratings.Take(10).Reverse().ToArray();

var chart = Chart2D.Chart.Bar<double, string, string, string, string>(
    values: top10Ratings,
    Keys: top10Languages,
    Name: "TIOBE Index"
)
.WithTitle("TIOBE Programming Community Index")
.WithLayout(Layout.init<string>(
    Width: 500,
    Height: 320,
    Title: Title.init("TIOBE Programming Community Index", X: 0.5, Y: 0.95),
    Margin: Margin.init<int, int, int, int, int, int>(Top: 60, Right: 20, Bottom: 50, Left: 120)
)).WithXAxis(LinearAxis.init<string, string, string, string, string, string, string, string>(
    Title: Title.init("Rating (%)")
))
.WithYAxis(LinearAxis.init<string, string, string, string, string, string, string, string>(
))
.WithConfig(Config.init(Responsive: false));
// chart.Show();
chart.Display();


<!-- Plotly chart will be drawn inside this DIV -->

[1] Andrew N. Harrington & George Thiruvathukal. (2021). Introduction to Computer Science in C#. <https://introcs.cs.luc.edu>
